# Live — Momentum del rango de apertura vs VWAP 24 h, Binance USDⓈ-M

Cuaderno de orquestación en vivo para `IntradayMomentumStrategy`, con la misma estructura que
`Turtle_MR_Intraday_Live_Binance.ipynb` (mismo bus de eventos, mismo `trade()`, mismo patrón de
Monitor/journal). Lo que cambia respecto a la estrategia de reversión:

| | Reversión (MR) | Momentum de apertura |
|---|---|---|
| Vela | 30 m | **5 m** |
| Señal | Ruptura Donchian + barra de confirmación (3 instantes) | Primera vela de la sesión vs VWAP 24 h (**2 instantes**) |
| Histórico necesario | ~21 barras | **≥ 289 barras** (ancla y N de 24 h) |
| Datos que consume | Solo lo que llega | **El marco 24/7 completo** — el ancla incluye la noche |
| Frecuencia | Cuando haya ruptura | **Una vez por sesión y símbolo**, a las 09:35 ET |
| Salida | Stop / objetivo / barrera vertical | **Reloj: 15:45 ET**, sin posición overnight |
| Stop | `stop_n·N` real | Sin stop en el backtest; **aquí sí hay uno, ancho** (ver §1.1) |

## 0. Veredicto del backtest — LEER ANTES DE TOCAR `RUN_LIVE`

`Intraday_Momentum_OR_VWAP.ipynb` es la validación de esta configuración sobre 4,7 años de
datos reales de Binance (1 225 sesiones, velas de 5 min). Su veredicto explícito:

> **NO DESPLIEGUES CAPITAL** — falla los **4 criterios críticos** (DSR, supervivencia a
> costes, R de cartera > 0, significancia del walk-forward), con la higiene de la medición
> pasando entera (12/12). La tasa base es una moneda al aire: el
> cierre de la sesión cae del lado de la señal el **49,8 %** de las veces en BTCUSDT. El poco
> bruto que hay se lo comen los costes (46 % en BTC, 77 % en ETH, **99 % en QQQUSDT** — el
> control que debería mostrar el efecto más fuerte si el mecanismo fuera real). La cartera de
> los tres símbolos pierde: CAGR −2,9 %, Sharpe −1,99. Con costes cero el flagship daría +6 %
> CAGR; en el escenario *optimista* (2 pb + 5 pb) ya es negativo. DSR = **0,0** con 70 ensayos.
> El diagnóstico es estructural, no de calibración: no hay nada que optimizar cuando la tasa
> base es 50 %.

Este cuaderno queda armado como **ejercicio técnico de integración** — sirve para validar que
el cableado datos → estrategia → portfolio → ejecución → monitor funciona de punta a punta
contra el testnet de Binance, NO para desplegar capital. `RUN_LIVE` se queda en `False`; y
aunque se active, debe apuntar SIEMPRE a `BINANCE_TESTNET_API_KEY` /
`BINANCE_TESTNET_SECRET_KEY`, nunca a producción.

Lo que sí vale de este cuaderno, independientemente del veredicto: es la primera estrategia del
repo que usa la salida por reloj de sesión (`session_flat_at`) y la separación entre unidad de
riesgo y orden de stop (`stop_n` / `stop_n_order`). Ambas piezas quedan en
`BinancePerpetualsPortfolio` con tests en `qsCrypto/portfolio/binance_session_test.py`, y son
reutilizables por cualquier estrategia intradía que venga después.

## 1. Imports y parámetros — declarados antes de abrir nada

In [ ]:
import json, logging, logging.config, os, queue, sys, threading, time
from datetime import datetime, timezone
from decimal import Decimal
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / "pyproject.toml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from qsCrypto.data.binance_klines import BinanceKlinePriceHandler
from qsCrypto.data.streaming import StreamingCryptoPrices
from qsCrypto.strategy.strategy import IntradayMomentumStrategy
from qsCrypto.portfolio.binance_portfolio import BinancePerpetualsPortfolio
from qsCrypto.execution.execution import create_execution_handler
from qsCrypto.monitor import Monitor
from qsCrypto.trading.trading import trade
from qsCrypto import settings

logging.config.fileConfig(ROOT / "qsCrypto" / "logging.conf")
logger = logging.getLogger("qsCrypto.notebooks.momentum_live")

### 1.1 Parámetros canónicos — idénticos al backtest, salvo uno

Los de la señal (`OR_BARS`, `VWAP_BARS`, `N_WINDOW`, la hora de apertura y la de cierre) son
**los mismos** que valida `Intraday_Momentum_OR_VWAP.ipynb`. Cambiarlos aquí y no allí rompe
la única garantía que tiene este sistema: que lo que se opera es lo que se midió.

La excepción deliberada es el **stop**. El backtest corre sin stop real (`mae_stop_n=inf`): es
una decisión de modelado, para poder observar la excursión adversa sin truncarla y calibrar el
stop a partir de ella. Una posición desatendida en un perpetuo **sin stop en el exchange** es
otra cosa: si el proceso muere, la red se cae o el websocket se queda mudo, nada la cierra. Así
que aquí sí se coloca uno, deliberadamente **ancho** (`STOP_N_LIVE`), lo bastante lejos para ser
inerte en condiciones normales — no es parte de la estrategia, es una red de seguridad
operativa. En el journal se verá si alguna vez se toca; si se toca a menudo, el backtest y la
operación han dejado de ser la misma cosa y hay que volver al cuaderno.

In [ ]:
SYMBOLS = ["BTCUSDT"]          # el flagship; ampliar solo tras revalidar

# ── SEÑAL: idénticos al backtest ───────────────────────────────────────────
BAR_INTERVAL  = "5m"
OR_BARS       = 1          # el rango de apertura = los primeros 5 minutos
VWAP_BARS     = 288        # 24 h de velas de 5 min, incluidas las overnight
N_WINDOW      = 288        # mismo horizonte para la N (rolling_range_n)
SESSION_TZ    = "America/New_York"
SESSION_OPEN  = "09:30"
SESSION_CLOSE = "15:45"    # <- la salida por reloj, en el portfolio
ALLOW_SHORT   = True

# ── RIESGO ─────────────────────────────────────────────────────────────────
RISK_PER_TRADE = Decimal("0.01")
MAX_GROSS_LEV  = 4.0
MAX_UNITS      = 1         # sin pirámide: la estrategia no la usa

# Red de seguridad operativa, NO parte de la estrategia (ver 1.1).
# El backtest no tiene stop; esto existe para que una posición desatendida no
# se quede sin nada que la cierre si el proceso muere.
STOP_N_LIVE = 4.0

# ── WARMUP ─────────────────────────────────────────────────────────────────
# min_bars = max(VWAP_BARS, N_WINDOW) + OR_BARS = 289. Se pide holgura: el
# handler descarta la vela en formación y una reconexión puede dejar huecos.
# El tope REST de Binance son 1500 velas, así que entra en una sola petición.
WARMUP_LIMIT = 800
assert WARMUP_LIMIT >= max(VWAP_BARS, N_WINDOW) + OR_BARS + 100

TEST_MODE = True           # SIEMPRE True hasta que haya veredicto positivo
EQUITY    = Decimal(str(settings.EQUITY))
HEARTBEAT = 0.5
MAX_ORDER_NOTIONAL_USDT = 200.0

JOURNAL_DIR = ROOT / "journals"; STATE_DIR = ROOT / "output"
SESSION_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
JOURNAL = JOURNAL_DIR / f"momentum_or_{SESSION_ID}.jsonl"
STATE = STATE_DIR / "live_state_momentum_or.json"
print("Parámetros fijados. Warmup:", WARMUP_LIMIT, "velas de", BAR_INTERVAL)

## 2. Cableado del motor

```
BinanceKlinePriceHandler    -> BarEvent (vela de 5 m cerrada, flag is_closed del kline)
StreamingCryptoPrices       -> TickEvent (bookTicker, para _ref_price/MAE)
IntradayMomentumStrategy    -> consume BAR del marco 24/7, emite UN SignalEvent por sesión
BinancePerpetualsPortfolio  -> consume SIGNAL, dimensiona por N, emite OrderEvent
                               y APLANA a las 15:45 ET (session_flat_at)
BinancePerpetualExecutionHandler -> MARKET + STOP_MARKET; puente de FillEvent
Monitor                     -> PortfolioEvent -> journal + snapshot atómico
```

Dos cosas que este cableado hace distinto del de reversión, y que son justo donde fallaría:

1. **La estrategia ve el marco 24/7.** No se le filtra la sesión: el ancla de 24 h y la N se
   alimentan de todas las velas, incluidas las de la noche. El filtro de sesión lo aplica la
   propia estrategia con su reloj (`_is_signal_bar`), no el flujo de datos.
2. **La salida es del portfolio, no de la estrategia.** `session_flat_at="15:45"` es lo que
   cierra la posición. Sin eso, nada la cerraría: `max_hold_bars` cuenta barras, no horas, y
   un hueco de datos lo descuadra.

In [ ]:
events = queue.Queue()

prices = BinanceKlinePriceHandler(
    pairs=SYMBOLS, events_queue=events, interval=BAR_INTERVAL,
    test_mode=TEST_MODE, market_type=settings.BINANCE_MARKET_TYPE,
    warmup_limit=WARMUP_LIMIT,
)
prices.resolve_specs()
prices.warmup()

ticks = StreamingCryptoPrices(
    pairs=SYMBOLS, events_queue=events, test_mode=TEST_MODE,
    market_type=settings.BINANCE_MARKET_TYPE,
)

strategy = IntradayMomentumStrategy(
    SYMBOLS, events, price_handler=prices,
    session_tz=SESSION_TZ, session_open=SESSION_OPEN,
    or_bars=OR_BARS, bar_seconds=300, vwap_bars=VWAP_BARS,
    n_window=N_WINDOW, allow_short=ALLOW_SHORT,
    max_history=WARMUP_LIMIT + 400,
)

execution = create_execution_handler(
    test_mode=TEST_MODE, market_type=settings.BINANCE_MARKET_TYPE,
    broker="binance", events_queue=events,
)

portfolio = BinancePerpetualsPortfolio(
    prices, events, home_currency="USDT", equity=EQUITY,
    risk_per_trade=RISK_PER_TRADE, backtest=False, specs=prices.specs,
    stop_n=1.0,                  # UNIDAD DE RIESGO: 1R = 1N, igual que el backtest
    stop_n_order=STOP_N_LIVE,    # dónde va el stop real: red de seguridad ancha (1.1)
    max_units=MAX_UNITS,
    max_hold_bars=10 ** 9,       # la barrera vertical no debe adelantarse al reloj
    tp_mode="take_n", take_n=1e9,   # sin objetivo: se sale por reloj
    max_gross_lev=MAX_GROSS_LEV,
    session_flat_at=SESSION_CLOSE, session_tz=SESSION_TZ,
    execution=execution, strategy=strategy,
    max_order_notional=MAX_ORDER_NOTIONAL_USDT,
)
print("Motor cableado.")

### 2.1 Comprobaciones antes de arrancar

Cuatro cosas que, si están mal, no dan error: simplemente no operan, u operan otra estrategia.
Se comprueban explícitamente en vez de confiar en que el warmup "habrá ido bien".

In [ ]:
import pandas as pd
from qsCrypto.strategy.strategy import rolling_range_n, compute_vwap

ok = {}

# 1. ¿Hay histórico suficiente para que la señal pueda existir siquiera?
for s in SYMBOLS:
    have = len(strategy.history[s])
    ok[f"{s}: histórico >= min_bars"] = have >= strategy.min_bars()
    print(f"{s}: {have} velas sembradas (min_bars={strategy.min_bars()})")

# 2. ¿El ancla y la N salen finitas con ese histórico?
for s in SYMBOLS:
    bars = strategy.history[s]
    n = strategy._n(bars); a = strategy._vwap(bars)
    ok[f"{s}: N y ancla finitas"] = bool(n == n and a == a and n > 0 and a > 0)
    print(f"{s}: N={n:,.2f}  VWAP24h={a:,.2f}")

# 3. ¿El reloj de sesión reconoce la barra de las 09:30 ET, y solo ésa?
probe = pd.Timestamp("2026-01-05 14:30")      # 09:30 EST
probe_dst = pd.Timestamp("2026-06-01 13:30")  # 09:30 EDT
ok["reloj: 09:30 EST"] = strategy._is_signal_bar(probe)[0]
ok["reloj: 09:30 EDT"] = strategy._is_signal_bar(probe_dst)[0]
ok["reloj: 09:35 no dispara"] = not strategy._is_signal_bar(
    probe + pd.Timedelta(minutes=5))[0]
ok["reloj: domingo no dispara"] = not strategy._is_signal_bar(
    pd.Timestamp("2026-01-04 14:30"))[0]

# 4. ¿El portfolio aplana a las 15:45, en ambos husos?
ok["cierre: 15:40 EST aplana"] = portfolio._past_session_close(
    pd.Timestamp("2026-01-05 20:40"))
ok["cierre: 15:40 EDT aplana"] = portfolio._past_session_close(
    pd.Timestamp("2026-06-01 19:40"))
ok["cierre: 15:35 NO aplana"] = not portfolio._past_session_close(
    pd.Timestamp("2026-01-05 20:35"))

print()
for k, v in ok.items():
    print(f"  [{'OK ' if v else 'MAL'}] {k}")
assert all(ok.values()), "revisa lo que salga en MAL antes de seguir"
print("\nTodas las comprobaciones pasan.")

## 3. Reconciliación contra el broker (testnet)

Si el proceso se reinició con una posición abierta, se reconstruye desde `positionRisk` en vez
de operar a ciegas sobre un estado local vacío. Para esta estrategia hay un matiz: una posición
reconciliada **de una sesión anterior** no debería existir (se cierra a las 15:45). Si aparece
una, es una señal de que el cierre por reloj falló y hay que mirarlo antes de arrancar.

In [ ]:
portfolio.reconcile_from_broker(execution)
pos = list(portfolio.positions.keys())
print(f"Posiciones tras reconciliar: {pos}")
if pos:
    print("\n⚠️  Hay posición abierta. Esta estrategia cierra a las 15:45 ET:")
    print("    si no estamos dentro de una sesión, el cierre por reloj falló.")
    print("    Revisa el journal de la sesión anterior antes de arrancar.")
    display(portfolio.snapshot())

## 4. Journal de la jornada

In [ ]:
monitor = Monitor(JOURNAL, STATE, session_id=SESSION_ID)
monitor.write(
    "session_start", symbols=SYMBOLS, equity=float(EQUITY),
    venue="binance_usdm", testnet=TEST_MODE,
    strategy="IntradayMomentumStrategy",
    bar_interval=BAR_INTERVAL, or_bars=OR_BARS, vwap_bars=VWAP_BARS,
    n_window=N_WINDOW, session_open=SESSION_OPEN, session_close=SESSION_CLOSE,
    session_tz=SESSION_TZ, risk=float(RISK_PER_TRADE),
    stop_n_live=STOP_N_LIVE, max_units=MAX_UNITS,
    max_order_notional_usdt=MAX_ORDER_NOTIONAL_USDT,
    specs=prices.specs,
    backtest_verdict="NO DESPLIEGUES CAPITAL (falla los 4 criticos; tasa base 49.8%, "
                     "DSR=0.0) — ver Intraday_Momentum_OR_VWAP.ipynb. Esta sesión "
                     "es un ejercicio técnico de integración, no despliegue.",
)
print(f"Journal : {JOURNAL}")
print(f"Estado  : {STATE}")

## 5. Arrancar la sesión

⚠️ **Esta celda envía órdenes reales a la cuenta de TESTNET de Binance** (nunca producción
mientras `TEST_MODE=True`). Con `RUN_LIVE=False` queda todo montado sin enviar nada — que es
como debe quedar mientras el veredicto del backtest no sea positivo.

Nota operativa: el motor tiene que estar arrancado **antes de las 09:30 ET**. Si se arranca a
media sesión, el warmup siembra el histórico pero la barra de señal de hoy ya pasó y la
estrategia no abrirá hasta mañana — que es el comportamiento correcto, no un fallo.

In [ ]:
RUN_LIVE = False   # cámbialo a True SOLO para una prueba deliberada en testnet

if not RUN_LIVE:
    print("RUN_LIVE=False — el motor no arranca. Nada se envía a Binance.")
else:
    if not TEST_MODE:
        raise RuntimeError(
            "TEST_MODE=False sin veredicto positivo del backtest: no se arranca "
            "contra producción.")
    trade_thread = threading.Thread(
        target=trade, args=(events, strategy, portfolio, execution, HEARTBEAT, monitor),
        daemon=True, name="trade-loop")
    kline_thread = threading.Thread(target=prices.stream_to_queue, daemon=True,
                                    name="binance-klines")
    tick_thread = threading.Thread(target=ticks.stream_to_queue, daemon=True,
                                   name="binance-ticks")
    userdata_thread = threading.Thread(target=execution.start_user_data_stream,
                                       daemon=True, name="binance-userdata")
    for t in (trade_thread, kline_thread, tick_thread, userdata_thread):
        t.start()
    print("Hilos arrancados: trade-loop, binance-klines, binance-ticks, binance-userdata")

## 6. Monitor en vivo

Además del snapshot habitual, muestra lo específico de esta estrategia: si la señal de hoy ya
se disparó, cuánto falta para el cierre por reloj y por qué se descartaron las sesiones sin
operación (`strategy.rejected`).

In [ ]:
from IPython.display import display, clear_output
import pandas as pd

MONITOR_INTERVAL_S = 5

def render_once():
    clear_output(wait=True)
    now = pd.Timestamp.utcnow().tz_localize(None)
    loc = (pd.Timestamp(now).tz_localize("UTC").tz_convert(SESSION_TZ))
    print(f"Sesión {SESSION_ID} | {loc:%Y-%m-%d %H:%M:%S} {SESSION_TZ}")
    print(f"hilos: trade={trade_thread.is_alive() if RUN_LIVE else '-'} "
          f"klines={kline_thread.is_alive() if RUN_LIVE else '-'} "
          f"ticks={tick_thread.is_alive() if RUN_LIVE else '-'}")
    print(f"último pulso: {monitor.last_heartbeat}")
    print()
    print("señal de hoy ya disparada:", strategy.fired)
    print("descartes acumulados    :", strategy.rejected)
    print("velas en el búfer       :",
          {s: len(strategy.history[s]) for s in SYMBOLS})
    hh, mm = (int(x) for x in SESSION_CLOSE.split(":"))
    close_local = loc.normalize() + pd.Timedelta(hours=hh, minutes=mm)
    print(f"cierre por reloj        : {SESSION_CLOSE} "
          f"({(close_local - loc).total_seconds()/60:+.0f} min)")
    print()
    print("posiciones locales:", portfolio.snapshot())
    if RUN_LIVE:
        try:
            bp = [p for p in execution.get_position_info()
                  if abs(float(p.get("positionAmt", 0))) > 0]
            print("posiciones en el broker (testnet):", bp)
        except Exception as exc:
            print(f"no se pudo leer el broker: {exc}")

# render_once()   # descomenta para una foto puntual
# while RUN_LIVE:
#     render_once(); time.sleep(MONITOR_INTERVAL_S)

## 7. Cierre de la jornada (kill switch)

Cancela lo vivo y aplana ANTES de cerrar la conexión — mismo orden y misma razón que en los
otros cuadernos: un stop todavía vivo tras aplanar podría reabrir posición contraria.

Para esta estrategia el kill switch es **redundante en condiciones normales**: el portfolio ya
aplana a las 15:45. Está aquí para lo anormal — parar a media sesión, o limpiar después de que
el cierre por reloj haya fallado.

In [ ]:
def kill_all(reason="fin_de_sesion"):
    for s in SYMBOLS:
        try:
            flat = execution.flatten(s, reason=reason)
            monitor.write("kill", symbol=s, flat=flat, reason=reason)
            print(f"  {s}: aplanado {flat:+g}")
        except Exception as exc:
            print(f"  {s}: fallo al aplanar — {exc}")
    execution.stop_user_data_stream()
    prices.stop()
    ticks.stop()

# kill_all()   # descomenta para aplanar todo ahora mismo

## 8. Extensión a IBKR / CME MBT — pendiente, no construido

La estrategia es, en su origen, una tesis sobre la sesión de contado de EE. UU., así que su
sitio natural son los futuros de bitcoin de CME (MBT) tanto como un perpetuo de Binance. Las
piezas ya existen y son simétricas a las de aquí:

| | Binance | IBKR / CME |
|---|---|---|
| Datos | `BinanceKlinePriceHandler` | `IBKRPriceHandler` |
| Portfolio | `BinancePerpetualsPortfolio` | `IBKRFuturesPortfolio` |
| Ejecución | `BinancePerpetualExecutionHandler` | `IBKRExecutionHandler` |
| Sesión | reloj propio (`_past_session_close`) | `qsCrypto/ibkr/sessions.py` (`TradingSessions.is_open`), con el calendario real de CME |

Dos razones para no construirlo todavía:

1. **Entitlement de datos.** La sonda del 2026-09-11 dejó los ticks de IBIT bloqueados
   (errores 10197/10089). Hasta resolverlo no hay datos en vivo con los que operar esto en IBKR.
2. `IBKRFuturesPortfolio` necesitaría el mismo `session_flat_at` que se ha añadido aquí a
   `BinancePerpetualsPortfolio` — con la ventaja de que en IBKR el calendario de sesiones es
   real (festivos y medias sesiones incluidos), en vez del reloj de pared que se usa aquí.

Ventaja estructural de la versión CME, cuando se pueda: **no hay funding**, y la sesión
regular del futuro coincide con la que la estrategia postula como mecanismo.